In [1]:
list.of.packages <- c("ggplot2", "Rcpp", "grf", "caret", "mltools", "rpart", "minpack.lm", "doParallel", "rattle", "anytime","rlist")
list.of.packages <- c(list.of.packages, "zoo", "dtw", "foreach", "evaluate","rlist","data.table","plyr","dplyr")


new.packages <- list.of.packages[!(list.of.packages %in% installed.packages()[,"Package"])]
if(length(new.packages)) install.packages(new.packages, lib='/home/zwang937/local/R_libs', repos="http://cran.us.r-project.org", dependencies = TRUE, INSTALL_opts = '--no-lock')


lapply(list.of.packages, require, character.only = TRUE)

Loading required package: ggplot2

Loading required package: Rcpp

Warning message:
“package ‘Rcpp’ was built under R version 4.3.2”
Loading required package: grf

Loading required package: caret

Loading required package: lattice

Error: package or namespace load failed for ‘caret’ in loadNamespace(i, c(lib.loc, .libPaths()), versionCheck = vI[[i]]):
 there is no package called ‘recipes’

Loading required package: mltools

Loading required package: rpart

Loading required package: minpack.lm

Loading required package: doParallel

Loading required package: foreach

Loading required package: iterators

Loading required package: parallel

Loading required package: rattle

Loading required package: tibble

Warning message:
“package ‘tibble’ was built under R version 4.3.2”
Loading required package: bitops

Rattle: A free graphical interface for data science with R.
Version 5.5.1 Copyright (c) 2006-2021 Togaware Pty Ltd.
Type 'rattle()' to shake, rattle, and roll your data.

Loading requir

[[1]]
[1] TRUE

[[2]]
[1] TRUE

[[3]]
[1] TRUE

[[4]]
[1] FALSE

[[5]]
[1] TRUE

[[6]]
[1] TRUE

[[7]]
[1] TRUE

[[8]]
[1] TRUE

[[9]]
[1] TRUE

[[10]]
[1] TRUE

[[11]]
[1] TRUE

[[12]]
[1] TRUE

[[13]]
[1] TRUE

[[14]]
[1] TRUE

[[15]]
[1] TRUE

[[16]]
[1] TRUE

[[17]]
[1] TRUE

[[18]]
[1] TRUE

[[19]]
[1] TRUE

In [2]:
# SET PARAMS
windowsize=2
num_trees = 200


print("Reading in data")
destfile <- paste("imputed_shifted_block.csv")
augmented_panel_data <- as.data.frame(fread(file = destfile))
# Time variant
#hhs_X_w_clusters_fpath = "../benchmark_tcv_kmeans_code/hhs_X_w_clusters.csv"
#hhs_X_w_clusters = as.data.frame(fread(file = hhs_X_w_clusters_fpath))

start_day = min(augmented_panel_data$days_from_start)
end_day = max(augmented_panel_data$days_from_start)

cutoff_list <- start_day:end_day

[1] "Reading in data"


In [3]:
augmented_panel_data

date,county,state,cases,deaths,datetime,days_from_start,rolled_cases,LAT,LON,⋯,Variant % XBB,Variant % XBB.1.16,Variant % XBB.1.5,Variant % XBB.1.5.1,Variant % XBB.1.9.1,Variant % XBB.1.9.2,Variant % XBB.2.3,fips,log_rolled_cases,log_shifted_rolled_cases
<IDate>,<chr>,<chr>,<dbl>,<dbl>,<IDate>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2020-04-17,Autauga,Alabama,20,2,2020-04-17,87,20.71429,32.53224,-86.64644,⋯,0,0,0,0,0,0,0,1001,3.030824,0.020906685
2020-04-18,Autauga,Alabama,19,2,2020-04-18,88,20.71429,32.53224,-86.64644,⋯,0,0,0,0,0,0,0,1001,3.030824,0.000000000
2020-04-19,Autauga,Alabama,21,2,2020-04-19,89,21.00000,32.53224,-86.64644,⋯,0,0,0,0,0,0,0,1001,3.044522,0.013698844
2020-04-20,Autauga,Alabama,22,1,2020-04-20,90,21.42857,32.53224,-86.64644,⋯,0,0,0,0,0,0,0,1001,3.064725,0.020202707
2020-04-21,Autauga,Alabama,23,1,2020-04-21,91,21.42857,32.53224,-86.64644,⋯,0,0,0,0,0,0,0,1001,3.064725,0.000000000
2020-04-22,Autauga,Alabama,25,2,2020-04-22,92,21.57143,32.53224,-86.64644,⋯,0,0,0,0,0,0,0,1001,3.071370,0.006644543
2020-04-23,Autauga,Alabama,23,2,2020-04-23,93,21.85714,32.53224,-86.64644,⋯,0,0,0,0,0,0,0,1001,3.084528,0.013158085
2020-04-24,Autauga,Alabama,26,2,2020-04-24,94,22.71429,32.53224,-86.64644,⋯,0,0,0,0,0,0,0,1001,3.122994,0.038466281
2020-04-25,Autauga,Alabama,25,2,2020-04-25,95,23.57143,32.53224,-86.64644,⋯,0,0,0,0,0,0,0,1001,3.160035,0.037041272


In [9]:
cutoff=200

print(paste0("Computing LLF for ", toString(cutoff)))
# START TRY

start_time <- Sys.time()
# Given my current cutoff, which block numbers should I use?
fname <- paste("llf_",toString(cutoff),".csv",sep="")        
# SLICE THE DATA ACCORDINGLY
early_data <- augmented_panel_data[augmented_panel_data$days_from_start <= cutoff, ]
early_data <- early_data[early_data$days_from_start >= max(cutoff-90, 0), ]

cutoff_parity <- cutoff %% 2

# Filter the dataframe based on the parity of t
early_data <- early_data %>%
  filter((days_from_start %% 2) == cutoff_parity)


case_number_columns <- c("fips","date", "datetime","county","state","days_from_start", "shifted_log_rolled_cases","log_rolled_cases")
early_data_case_numbers <- early_data[, names(early_data) %in% case_number_columns]
# Format data to be fed to ll_regression_forest
#WYX <- merge(early_data_case_numbers, X_time_invariant, by="fips", all.x=TRUE)
XY <- early_data
XY <- XY[order(XY$fips, XY$days_from_start),]

Y <- XY$log_shifted_rolled_cases
X <- XY[, !names(XY) %in% case_number_columns]
X <- X %>%
  select_if(is.numeric)


[1] "Computing LLF for 200"


In [8]:
XY

,date,county,state,cases,deaths,datetime,days_from_start,rolled_cases,LAT,LON,⋯,Variant % XBB,Variant % XBB.1.16,Variant % XBB.1.5,Variant % XBB.1.5.1,Variant % XBB.1.9.1,Variant % XBB.1.9.2,Variant % XBB.2.3,fips,log_rolled_cases,log_shifted_rolled_cases
,<IDate>,<chr>,<chr>,<dbl>,<dbl>,<IDate>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,2020-05-10,Autauga,Alabama,49,4,2020-05-10,110,38.85714,32.53224,-86.64644,⋯,0,0,0,0,0,0,0,1001,3.659892,0.076372979
2,2020-05-12,Autauga,Alabama,63,4,2020-05-12,112,46.28571,32.53224,-86.64644,⋯,0,0,0,0,0,0,0,1001,3.834833,0.093768159
3,2020-05-14,Autauga,Alabama,72,4,2020-05-14,114,55.42857,32.53224,-86.64644,⋯,0,0,0,0,0,0,0,1001,4.015095,0.097374164
4,2020-05-16,Autauga,Alabama,74,4,2020-05-16,116,64.00000,32.53224,-86.64644,⋯,0,0,0,0,0,0,0,1001,4.158883,0.074107972
5,2020-05-18,Autauga,Alabama,83,4,2020-05-18,118,71.14286,32.53224,-86.64644,⋯,0,0,0,0,0,0,0,1001,4.264690,0.053621091
6,2020-05-20,Autauga,Alabama,96,3,2020-05-20,120,79.42857,32.53224,-86.64644,⋯,0,0,0,0,0,0,0,1001,4.374858,0.061186830
7,2020-05-22,Autauga,Alabama,107,3,2020-05-22,122,89.28571,32.53224,-86.64644,⋯,0,0,0,0,0,0,0,1001,4.491842,0.061024702
8,2020-05-24,Autauga,Alabama,114,3,2020-05-24,124,100.71429,32.53224,-86.64644,⋯,0,0,0,0,0,0,0,1001,4.612288,0.059915653
9,2020-05-26,Autauga,Alabama,136,3,2020-05-26,126,112.85714,32.53224,-86.64644,⋯,0,0,0,0,0,0,0,1001,4.726123,0.062683702


In [10]:
llf <- ll_regression_forest(X,Y, num.trees=num_trees)


In [11]:
llf

GRF forest object of type ll_regression_forest 
Number of trees: 200 
Number of training samples: 74826 
Variable importance: 
    1     2     3     4     5     6     7     8     9    10    11    12    13 
0.063 0.035 0.004 0.006 0.010 0.000 0.000 0.001 0.000 0.001 0.001 0.000 0.000 
   14    15    16    17    18    19    20    21    22    23    24    25    26 
0.000 0.000 0.000 0.000 0.001 0.001 0.000 0.001 0.001 0.000 0.000 0.000 0.000 
   27    28    29    30    31    32    33    34    35    36    37    38    39 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
   40    41    42    43    44    45    46    47    48    49    50    51    52 
0.000 0.000 0.000 0.001 0.000 0.000 0.000 0.001 0.001 0.000 0.000 0.000 0.000 
   53    54    55    56    57    58    59    60    61    62    63    64    65 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
   66    67    68    69    70    71    72    73    74    75    76    77    78 
0.00